# 03 · LSTM — Previsão de Séries Temporais

Modela o **preço de fechamento** de cada criptomoeda como série temporal univariada e treina uma **LSTM single-layer (não-stacked)** para prever o próximo dia.

### Especificações da arquitetura

| Hiperparâmetro | Valor |
|----------------|-------|
| `look_back_window` | 90 dias |
| `units` | 100 |
| `dropout` | 0.2 |
| `batch_size` | 32 |
| `epochs` (máx) | 200 |
| `EarlyStopping patience` | 15 |
| Loss | MSE |
| Otimizador | Adam (lr=0.001) |
| Métricas finais | RMSE e MAE em USD |

---

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
from src.data.loader import load_coin
from src.data.splitter import temporal_split
from src.features.timeseries_features import prepare_lstm_data, inverse_scale
from src.models.lstm_model import LSTMModel
from src.evaluation.regression_metrics import evaluate_regressor, print_regression_report
from src.visualization.lstm_plots import (
    plot_training_history,
    plot_predictions_timeseries,
    plot_error_distribution,
)

## 1. Função de Treinamento por Ticker

Encapsulamos o pipeline completo em uma função que recebe um ticker e retorna o modelo treinado, as predições em dólares e as métricas.

In [ ]:
def train_lstm_for_ticker(ticker: str, look_back: int = 90, units: int = 100,
                           batch_size: int = 32, epochs: int = 200,
                           patience: int = 15, verbose: int = 1) -> dict:
    print(f'\n{"═"*60}\n  Treinando LSTM para {ticker}\n{"═"*60}')

    # 1. Dados
    df = load_coin(ticker)
    prep = prepare_lstm_data(df, price_col='Close', look_back=look_back)
    X, y, scaler = prep['X'], prep['y'], prep['scaler']

    # 2. Split temporal (sem embaralhamento)
    X_train, X_test, y_train, y_test = temporal_split(X, y, test_size=0.20)

    # 3. Modelo
    lstm = LSTMModel(
        look_back=look_back, units=units,
        batch_size=batch_size, epochs=epochs,
        patience=patience, ticker=ticker,
    )
    lstm.fit(X_train, y_train, verbose=verbose)

    # 4. Predição + inversão de escala (de [0,1] para dólares)
    y_pred_scaled = lstm.predict(X_test)
    y_pred_dollars = inverse_scale(y_pred_scaled, scaler)
    y_true_dollars = inverse_scale(y_test, scaler)

    # 5. Métricas em dólares
    metrics = evaluate_regressor(y_true_dollars, y_pred_dollars, in_dollars=True)
    print_regression_report(metrics, model_name=f'LSTM — {ticker}')

    return {
        'ticker': ticker,
        'model':  lstm,
        'history': lstm.history,
        'y_true_dollars': y_true_dollars,
        'y_pred_dollars': y_pred_dollars,
        'scaler': scaler,
        'metrics': metrics,
    }

## 2. Treinar BTC (exemplo detalhado)

In [ ]:
resultado_btc = train_lstm_for_ticker('BTC')

### 2.1 Curvas de Treinamento

In [ ]:
plot_training_history(
    resultado_btc['history'],
    title='Curvas de Treinamento — LSTM BTC',
    save_as='lstm_btc_history.png',
)

### 2.2 Predição vs Real (em USD)

In [ ]:
plot_predictions_timeseries(
    resultado_btc['y_true_dollars'],
    resultado_btc['y_pred_dollars'],
    ticker='BTC',
    save_as='lstm_btc_predicoes.png',
)

### 2.3 Distribuição do Erro (em USD)

In [ ]:
plot_error_distribution(
    resultado_btc['y_true_dollars'],
    resultado_btc['y_pred_dollars'],
    ticker='BTC',
    save_as='lstm_btc_erros.png',
)

## 3. Treinar Demais Tickers

Treina ETH, XRP e DASH na sequência. Cada modelo é salvo automaticamente em `outputs/models/lstm_<TICKER>.keras`.

In [ ]:
resultados = {'BTC': resultado_btc}

for ticker in ['ETH', 'XRP', 'DASH']:
    resultados[ticker] = train_lstm_for_ticker(ticker, verbose=0)

## 4. Salvar Modelos

In [ ]:
for ticker, r in resultados.items():
    r['model'].save()
    print(f'✓ Modelo salvo: lstm_{ticker}.keras')

## 5. Comparação Entre Tickers

In [ ]:
rows = []
for ticker, r in resultados.items():
    m = r['metrics']
    rows.append({
        'ticker': ticker,
        'r2':     m['r2'],
        'mae_usd':  m['mae'],
        'rmse_usd': m['rmse'],
        'mape_pct': m['mape'] * 100 if m['mape'] is not None else None,
    })

df_lstm = pd.DataFrame(rows)
df_lstm

In [ ]:
# Visualização agregada
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (ticker, r) in zip(axes.flat, resultados.items()):
    ax.plot(r['y_true_dollars'], label='Real',    color='#1976D2', linewidth=1.3)
    ax.plot(r['y_pred_dollars'], label='Predito', color='#E64A19', linewidth=1.3, alpha=0.85)
    rmse = r['metrics']['rmse']
    mae  = r['metrics']['mae']
    ax.set_title(f'{ticker} | RMSE=${rmse:,.2f} | MAE=${mae:,.2f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Índice (teste)')
    ax.set_ylabel('Preço (USD)')
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)

plt.suptitle('Predições LSTM — Todas as Criptomoedas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/lstm_todas_predicoes.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Conclusões

### Pontos a discutir na apresentação

**Por que LSTM e não CNN?**  
LSTM é especializada em capturar dependências temporais de longo prazo. Para previsão de preços, padrões como tendências, ciclos e regimes de volatilidade se manifestam em escalas que vão de dias a semanas — exatamente o que LSTM modela bem com sua célula de memória. CNN seria mais adequada para detecção de padrões locais em janelas fixas.

**Por que single-layer?**  
Para séries financeiras com forte componente estocástico (ruído de mercado), modelos profundos (stacked LSTM) tendem a overfittar. Um modelo enxuto com regularização via dropout costuma generalizar melhor.

**Métricas em dólares — interpretação**  
O MAE em USD diz, em média, em quantos dólares o modelo erra a previsão do próximo dia. Para BTC (preços ~50k), um MAE de $1.500 é ~3% do preço. Para XRP (~$0.50), $0.05 já representa 10%. Por isso o **MAPE** é a métrica mais comparável entre tickers.

**Limitações**  
- Modelo univariado (usa só preço de fechamento). Adicionar volume, retorno, indicadores   técnicos provavelmente melhoraria as predições.
- Predição t+1: para horizontes maiores (t+7, t+30), o erro acumula rapidamente.
- Não captura eventos exógenos (notícias, regulação) — um modelo multimodal seria melhor.